In [1]:
import os
import pandas as pd
import snowflake.connector

In [2]:
DATABASE = os.getenv("SNOWFLAKE_DATABASE")
ANALYTICS_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_ANALYTICS")

STAGE_TABLE = "TRIPS_ENRICHED_UNIFIED_STAGE"
OBT_TABLE = "OBT_TRIPS"

RESULTS_03 = []

In [3]:
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=DATABASE,
    role=os.getenv("SNOWFLAKE_ROLE"),
)

cur = conn.cursor()
cur.execute(f"USE DATABASE {DATABASE}")
cur.execute(f"CREATE SCHEMA IF NOT EXISTS {ANALYTICS_SCHEMA}")

print("Snowflake connection ready")

Snowflake connection ready


In [4]:
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {ANALYTICS_SCHEMA}.{OBT_TABLE} (
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    pickup_date DATE,
    pickup_hour INTEGER,
    dropoff_date DATE,
    dropoff_hour INTEGER,
    day_of_week INTEGER,
    month INTEGER,
    year INTEGER,

    pu_location_id INTEGER,
    pu_zone STRING,
    pu_borough STRING,
    do_location_id INTEGER,
    do_zone STRING,
    do_borough STRING,

    service_type STRING,
    vendor_id INTEGER,
    vendor_name STRING,
    rate_code_id INTEGER,
    rate_code_desc STRING,
    payment_type INTEGER,
    payment_type_desc STRING,
    trip_type INTEGER,

    passenger_count FLOAT,
    trip_distance FLOAT,
    store_and_fwd_flag STRING,

    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    tolls_amount FLOAT,
    improvement_surcharge FLOAT,
    congestion_surcharge FLOAT,
    airport_fee FLOAT,
    ehail_fee FLOAT,
    total_amount FLOAT,

    trip_duration_min FLOAT,
    avg_speed_mph FLOAT,
    tip_pct FLOAT,

    run_id STRING,
    ingested_at_utc TIMESTAMP,
    source_year INTEGER,
    source_month INTEGER
)
""")

print("OBT table ready")

OBT table ready


In [5]:
def delete_obt_month(service_type, year_value, month_value):
    cur.execute(f"""
        DELETE FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE service_type = '{service_type}'
          AND source_year = {year_value}
          AND source_month = {month_value}
    """)

In [6]:
def build_obt_month(service_type, year_value, month_value):
    delete_obt_month(service_type, year_value, month_value)

    sql = f"""
    INSERT INTO {ANALYTICS_SCHEMA}.{OBT_TABLE}
    SELECT
        pickup_datetime,
        dropoff_datetime,
        CAST(pickup_datetime AS DATE) AS pickup_date,
        EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
        CAST(dropoff_datetime AS DATE) AS dropoff_date,
        EXTRACT(HOUR FROM dropoff_datetime) AS dropoff_hour,
        DAYOFWEEK(pickup_datetime) AS day_of_week,
        EXTRACT(MONTH FROM pickup_datetime) AS month,
        EXTRACT(YEAR FROM pickup_datetime) AS year,

        pu_location_id,
        pu_zone,
        pu_borough,
        do_location_id,
        do_zone,
        do_borough,

        service_type,
        vendor_id,
        vendor_name,
        rate_code_id,
        rate_code_desc,
        payment_type,
        payment_type_desc,
        trip_type,

        passenger_count,
        trip_distance,
        store_and_fwd_flag,

        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        congestion_surcharge,
        airport_fee,
        ehail_fee,
        total_amount,

        ROUND(DATEDIFF('second', pickup_datetime, dropoff_datetime) / 60.0, 2) AS trip_duration_min,

        CASE
            WHEN DATEDIFF('second', pickup_datetime, dropoff_datetime) > 0
                 AND trip_distance > 0
            THEN ROUND(
                trip_distance / (DATEDIFF('second', pickup_datetime, dropoff_datetime) / 3600.0),
                2
            )
            ELSE NULL
        END AS avg_speed_mph,

        CASE
            WHEN fare_amount > 0
            THEN ROUND((tip_amount / fare_amount) * 100.0, 2)
            ELSE NULL
        END AS tip_pct,

        run_id,
        ingested_at_utc,
        source_year,
        source_month

    FROM {ANALYTICS_SCHEMA}.{STAGE_TABLE}
    WHERE service_type = '{service_type}'
      AND source_year = {year_value}
      AND source_month = {month_value}
      AND pickup_datetime IS NOT NULL
      AND dropoff_datetime IS NOT NULL
      AND trip_distance > 0
      AND total_amount >= 0
      AND DATEDIFF('second', pickup_datetime, dropoff_datetime) >= 0
    """
    
    cur.execute(sql)

In [7]:
build_obt_month("yellow", 2024, 1)
build_obt_month("green", 2024, 1)

print("Test month loaded into OBT")

Test month loaded into OBT


In [8]:
test_queries = {
    "yellow_2024_01_count": f"""
        SELECT COUNT(*)
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE service_type = 'yellow'
          AND source_year = 2024
          AND source_month = 1
    """,
    "green_2024_01_count": f"""
        SELECT COUNT(*)
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE service_type = 'green'
          AND source_year = 2024
          AND source_month = 1
    """,
    "sample_rows": f"""
        SELECT
            service_type,
            pickup_datetime,
            dropoff_datetime,
            trip_distance,
            total_amount,
            trip_duration_min,
            avg_speed_mph,
            tip_pct
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE source_year = 2024
          AND source_month = 1
        LIMIT 10
    """
}

for name, query in test_queries.items():
    print(f"\n--- {name} ---")
    cur.execute(query)
    for row in cur.fetchmany(10):
        print(row)


--- yellow_2024_01_count ---
(2872094,)

--- green_2024_01_count ---
(53575,)

--- sample_rows ---
('yellow', datetime.datetime(2024, 1, 25, 21, 21, 43), datetime.datetime(2024, 1, 25, 21, 32, 56), 2.1, 18.5, 11.22, 11.23, 0.0)
('yellow', datetime.datetime(2024, 1, 25, 21, 36, 6), datetime.datetime(2024, 1, 25, 21, 54, 29), 1.57, 25.56, 18.38, 5.12, 26.13)
('yellow', datetime.datetime(2024, 1, 25, 21, 23, 33), datetime.datetime(2024, 1, 25, 21, 41, 7), 2.37, 26.4, 17.57, 8.09, 25.88)
('yellow', datetime.datetime(2024, 1, 25, 21, 30, 28), datetime.datetime(2024, 1, 25, 21, 41, 40), 2.31, 19.5, 11.2, 12.37, 7.41)
('yellow', datetime.datetime(2024, 1, 25, 21, 38, 39), datetime.datetime(2024, 1, 25, 21, 48, 21), 1.09, 18.0, 9.7, 6.74, 30.0)
('yellow', datetime.datetime(2024, 1, 25, 21, 40, 13), datetime.datetime(2024, 1, 25, 22, 4, 17), 11.59, 65.59, 24.07, 28.89, 23.28)
('yellow', datetime.datetime(2024, 1, 25, 21, 39, 21), datetime.datetime(2024, 1, 25, 21, 44, 13), 0.98, 13.2, 4.87, 12

In [9]:
for service_type in ["yellow", "green"]:
    for year_value in range(2015, 2026):
        for month_value in range(1, 13):
            try:
                print(f"Building OBT for {service_type} {year_value}-{month_value:02d}")
                build_obt_month(service_type, year_value, month_value)

                RESULTS_03.append({
                    "service_type": service_type,
                    "year": year_value,
                    "month": month_value,
                    "status": "success",
                    "message": "ok"
                })

                print(f"Done OBT for {service_type} {year_value}-{month_value:02d}")

            except Exception as e:
                RESULTS_03.append({
                    "service_type": service_type,
                    "year": year_value,
                    "month": month_value,
                    "status": "failed",
                    "message": str(e)
                })

                print(f"Failed OBT for {service_type} {year_value}-{month_value:02d}: {e}")

Building OBT for yellow 2015-01
Done OBT for yellow 2015-01
Building OBT for yellow 2015-02
Done OBT for yellow 2015-02
Building OBT for yellow 2015-03
Done OBT for yellow 2015-03
Building OBT for yellow 2015-04
Done OBT for yellow 2015-04
Building OBT for yellow 2015-05
Done OBT for yellow 2015-05
Building OBT for yellow 2015-06
Done OBT for yellow 2015-06
Building OBT for yellow 2015-07
Done OBT for yellow 2015-07
Building OBT for yellow 2015-08
Done OBT for yellow 2015-08
Building OBT for yellow 2015-09
Done OBT for yellow 2015-09
Building OBT for yellow 2015-10
Done OBT for yellow 2015-10
Building OBT for yellow 2015-11
Done OBT for yellow 2015-11
Building OBT for yellow 2015-12
Done OBT for yellow 2015-12
Building OBT for yellow 2016-01
Done OBT for yellow 2016-01
Building OBT for yellow 2016-02
Done OBT for yellow 2016-02
Building OBT for yellow 2016-03
Done OBT for yellow 2016-03
Building OBT for yellow 2016-04
Done OBT for yellow 2016-04
Building OBT for yellow 2016-05
Done OBT

In [10]:
results_03_df = pd.DataFrame(RESULTS_03)
results_03_df

,service_type,year,month,status,message
0,yellow,2015,1,success,ok
1,yellow,2015,2,success,ok
2,yellow,2015,3,success,ok
3,yellow,2015,4,success,ok
4,yellow,2015,5,success,ok
...,...,...,...,...,...
259,green,2025,8,success,ok
260,green,2025,9,success,ok
261,green,2025,10,success,ok
262,green,2025,11,success,ok


In [11]:
results_03_df.groupby(["service_type", "status"]).size()

service_type  status 
green         success    132
yellow        success    132
dtype: int64

In [12]:
results_03_df.to_csv("obt_build_results.csv", index=False)
print("Saved results to obt_build_results.csv")

Saved results to obt_build_results.csv


In [13]:
validation_queries = {
    "total_rows": f"""
        SELECT COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
    """,
    "rows_by_service": f"""
        SELECT service_type, COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        GROUP BY service_type
        ORDER BY service_type
    """,
    "rows_by_year": f"""
        SELECT service_type, source_year, COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        GROUP BY service_type, source_year
        ORDER BY service_type, source_year
    """,
    "sample_metrics": f"""
        SELECT
            service_type,
            trip_distance,
            total_amount,
            trip_duration_min,
            avg_speed_mph,
            tip_pct
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        LIMIT 20
    """
}

for name, query in validation_queries.items():
    print(f"\n--- {name} ---")
    cur.execute(query)
    for row in cur.fetchmany(20):
        print(row)


--- total_rows ---
(857782756,)

--- rows_by_service ---
('green', 66981035)
('yellow', 790801721)

--- rows_by_year ---
('green', 2015, 18937057)
('green', 2016, 16141551)
('green', 2017, 11579087)
('green', 2018, 8777350)
('green', 2019, 6128892)
('green', 2020, 1665538)
('green', 2021, 1026880)
('green', 2022, 789648)
('green', 2023, 746198)
('green', 2024, 624310)
('green', 2025, 564524)
('yellow', 2015, 145119381)
('yellow', 2016, 130312046)
('yellow', 2017, 112710727)
('yellow', 2018, 102111083)
('yellow', 2019, 83700266)
('yellow', 2020, 24214406)
('yellow', 2021, 30336930)
('yellow', 2022, 38848871)
('yellow', 2023, 37198342)

--- sample_metrics ---
('yellow', 9.4, 29.74, 14.3, 39.44, 0.0)
('yellow', 1.6, 9.8, 10.03, 9.57, 0.0)
('yellow', 1.8, 14.5, 16.75, 6.45, 19.13)
('yellow', 1.46, 10.4, 8.98, 9.75, 20.0)
('yellow', 1.29, 12.2, 13.42, 5.77, 20.0)
('yellow', 1.0, 12.3, 18.42, 3.26, 0.0)
('yellow', 1.65, 11.6, 11.08, 8.93, 20.0)
('yellow', 2.2, 16.56, 19.22, 6.87, 21.23)
('y

In [14]:
cur.close()
conn.close()
print("Notebook 03 finished")

Notebook 03 finished
